# ProstT5 Speculative Decoding — enc-CNN + Profile HMM Drafters (3Di → AA)

This notebook implements **actual speculative decoding** for ProstT5 inverse folding, using the enc-CNN as a fast drafter.

**Prerequisites:** Run `prostT5_baseline_performance.ipynb` first to generate baseline timings and α predictions.

## What this notebook does

1. **Custom speculative decoding loop** — Leviathan’s algorithm from scratch (greedy + stochastic)
2. **HuggingFace `assistant_model` wrapper** — CNN wrapped as a compatible assistant for `model.generate()`
3. **Benchmark** — Sweep draft lengths K ∈ {1, 3, 5, 8, 16}, measure real speedup
4. **Compare** — Predicted speedup (from α) vs. measured speedup

Key property: greedy speculative decoding produces **exactly the same output** as plain enc-dec greedy generation.

References:
- Leviathan et al. (2023). Fast Inference from Transformers via Speculative Decoding. ICML.
- HuggingFace blog: Assisted Generation (https://huggingface.co/blog/assisted-generation)

In [1]:
%pip install tiktoken sentencepiece torch
%pip install 'accelerate>=0.26.0'
%pip install "transformers==4.46.3" "protobuf>=3.20" sentencepiece

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 90.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 100.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
Note: 

In [2]:
#@title Google Drive mount + env vars. { display-mode: "form" }
import os

os.environ["USE_TF"] = "0"

try:
    # from google.colab import drive
    # drive.mount('/content/drive')
    os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
    DRIVE_ROOT = "/kaggle/working/prostT5_benchmarks"
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    USE_DRIVE = True
    print(f"Google Drive mounted. HF cache -> {os.environ['HF_HOME']}")
except (ImportError, ModuleNotFoundError):
    print("Not on Colab; using local paths.")
    DRIVE_ROOT = None
    USE_DRIVE = False

Google Drive mounted. HF cache -> /kaggle/working/hf_cache


In [3]:
#@title Imports. { display-mode: "form" }
import os, time, json, gc, statistics, pickle
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import T5Tokenizer, AutoModelForSeq2SeqLM, GenerationConfig

if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"torch={torch.__version__}  device={device}")
if device.type == "cpu":
    print("WARNING: running on CPU — timings will not reflect realistic GPU latency.")

torch=2.10.0+cu128  device=cuda:0


In [4]:
#@title Configuration. { display-mode: "form" }
PROSTT5_NAME = "Rostlab/ProstT5_fp16"
NOTEBOOK_DIR = Path(".").resolve()

if USE_DRIVE:
    PROJECT_DIR = Path(DRIVE_ROOT)
else:
    PROJECT_DIR = NOTEBOOK_DIR

CNN_CKPT = PROJECT_DIR / "model.pt"
if not CNN_CKPT.exists():
    for candidate in [NOTEBOOK_DIR / "model.pt",
                      NOTEBOOK_DIR / "cnn_chkpnt_AA_CNN" / "model.pt"]:
        if candidate.exists():
            CNN_CKPT = candidate
            break

RESULTS_DIR = NOTEBOOK_DIR / "spec_decode_results"
RESULTS_DIR.mkdir(exist_ok=True)

if USE_DRIVE:
    CHECKPOINT_DIR = Path(DRIVE_ROOT) / "spec_decode_checkpoints"
else:
    CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Draft lengths to sweep
K_VALUES = [1, 3, 5, 8, 16]

# Timing protocol
NUM_WARMUP = 2
NUM_REPEATS = 3
USE_FP16 = True

# AA vocabulary (alphabetical, matches CNN output classes)
AA_LETTERS = "ACDEFGHIKLMNPQRSTVWY"
assert len(AA_LETTERS) == 20

print(f"CNN checkpoint: {CNN_CKPT}  (exists: {CNN_CKPT.exists()})")
print(f"Results dir: {RESULTS_DIR}")
print(f"K values to sweep: {K_VALUES}")

CNN checkpoint: /kaggle/working/prostT5_benchmarks/model.pt  (exists: True)
Results dir: /kaggle/working/spec_decode_results
K values to sweep: [1, 3, 5, 8, 16]


In [5]:
#@title Load ProstT5 + CNN. { display-mode: "form" }
tokenizer = T5Tokenizer.from_pretrained(PROSTT5_NAME, do_lower_case=False, legacy=True)

dtype = torch.float16 if (USE_FP16 and device.type == "cuda") else torch.float32
model = AutoModelForSeq2SeqLM.from_pretrained(
    PROSTT5_NAME, low_cpu_mem_usage=True, torch_dtype=dtype,
).to(device).eval()
if dtype == torch.float16:
    model = model.half()

encoder = model.get_encoder()

class AACNN(nn.Module):
    def __init__(self, num_tokens=20, hidden=32, in_dim=1024):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Conv2d(in_dim, hidden, kernel_size=(7, 1), padding=(3, 0)),
            nn.ReLU(),
            nn.Dropout(0.0),
            nn.Conv2d(hidden, num_tokens, kernel_size=(7, 1), padding=(3, 0)),
        )

    def forward(self, x):
        x = x.permute(0, 2, 1).unsqueeze(-1)
        x = self.classifier(x)
        return x.squeeze(-1).permute(0, 2, 1)

cnn = AACNN(num_tokens=20).to(device).eval()
ckpt = torch.load(CNN_CKPT, map_location=device, weights_only=False)
state_dict = ckpt.get("state_dict", ckpt)
cnn.load_state_dict(state_dict, strict=True)

# Pre-compute AA token IDs in ProstT5 vocabulary
# ProstT5 generates space-separated AAs, so tokens are ▁A, ▁C, ... (not bare A, C, ...).
# convert_tokens_to_ids("A") returns <unk> (ID=2) — must use encode(" A") instead.
AA_TOKEN_IDS = [tokenizer.encode(f" {aa}", add_special_tokens=False)[0] for aa in AA_LETTERS]
AA_TOKEN_ID_TO_CNN_IDX = {tid: i for i, tid in enumerate(AA_TOKEN_IDS)}
CNN_IDX_TO_TOKEN_ID = {i: tid for i, tid in enumerate(AA_TOKEN_IDS)}

# Decoder start token
DECODER_START_TOKEN_ID = model.config.decoder_start_token_id

# Sanity check
for aa, tid in zip(AA_LETTERS[:3], AA_TOKEN_IDS[:3]):
    decoded = tokenizer.decode([tid]).strip()
    assert decoded == aa, f"Token ID mismatch: {aa} -> {tid} -> '{decoded}'"

print(f"ProstT5 loaded. params={sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"CNN loaded. params={sum(p.numel() for p in cnn.parameters()):,}")
print(f"Decoder start token ID: {DECODER_START_TOKEN_ID}")
print(f"AA token IDs (first 5): {list(zip(AA_LETTERS[:5], AA_TOKEN_IDS[:5]))}")

ProstT5 loaded. params=2818.9M
CNN loaded. params=233,908
Decoder start token ID: 0
AA token IDs (first 5): [('A', 3), ('C', 22), ('D', 10), ('E', 9), ('F', 15)]


In [6]:
#@title Load test set + baseline results. { display-mode: "form" }
import sys
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(NOTEBOOK_DIR))

try:
    from test_set_100 import TEST_SET, TEST_IDS, BACKUP_IDS
    print(f"Loaded {len(TEST_SET)} proteins from test_set_100.py")
except ImportError:
    print("test_set_100.py not found; using fallback.")
    TEST_IDS = [
        "P62990", "P01308", "P02798", "P62944", "P01542",
        "P43408", "P0AFL6", "P0ACJ8", "P0A988", "P0A7Y4",
    ]

# Load baseline checkpoint (has test_set with 3Di sequences)
baseline_ckpt_path = CHECKPOINT_DIR.parent / "checkpoints" / "checkpoint_latest.pkl"
if not baseline_ckpt_path.exists():
    baseline_ckpt_path = RESULTS_DIR.parent / "benchmark_results" / "checkpoints" / "checkpoint_latest.pkl"

# Try loading the test set from the FASTA files built by notebook 1
DATA_DIR = NOTEBOOK_DIR / "benchmark_data"
TEST_FASTA = DATA_DIR / "test_set_3Di.fasta"
TEST_AA_FASTA = DATA_DIR / "test_set_AA.fasta"

test_set = {}
if TEST_FASTA.exists() and TEST_AA_FASTA.exists():
    cur = None
    for line in TEST_AA_FASTA.read_text().splitlines():
        if line.startswith(">"):
            cur = line[1:].split()[0]
            test_set.setdefault(cur, {})["aa"] = ""
        elif cur:
            test_set[cur]["aa"] += line.strip()
    cur = None
    for line in TEST_FASTA.read_text().splitlines():
        if line.startswith(">"):
            cur = line[1:].split()[0]
            test_set.setdefault(cur, {})["3di"] = ""
        elif cur:
            test_set[cur]["3di"] += line.strip().lower()
    for uid, rec in test_set.items():
        rec["length"] = len(rec.get("3di", ""))
    print(f"Loaded test set from FASTA: {len(test_set)} proteins")
else:
    print("WARNING: FASTA files not found. Run baseline notebook first.")
    print(f"Expected at: {TEST_FASTA}")

# Load alpha predictions from baseline (if available)
alpha_predictions = {}
baseline_ckpt = None
for ckpt_path in [CHECKPOINT_DIR.parent / "checkpoints" / "checkpoint_latest.pkl",
                  Path(DRIVE_ROOT) / "checkpoints" / "checkpoint_latest.pkl" if DRIVE_ROOT else Path("/dev/null")]:
    if ckpt_path.exists():
        with open(ckpt_path, "rb") as f:
            baseline_ckpt = pickle.load(f)
        alpha_predictions = baseline_ckpt.get("alpha_results", {})
        print(f"Loaded baseline checkpoint: {len(alpha_predictions)} proteins with alpha data")
        break

if not alpha_predictions:
    print("No alpha predictions found (baseline not yet complete). Will skip comparison plots.")

Loaded 100 proteins from test_set_100.py
Loaded test set from FASTA: 97 proteins
Loaded baseline checkpoint: 100 proteins with alpha data


## Part 1 — Custom Speculative Decoding Implementation

Implements Leviathan’s algorithm from scratch. Supports greedy (deterministic, output identical to enc-dec) and stochastic (samples from same distribution as enc-dec) modes.

In [7]:
#@title Helper: format input + reference generation. { display-mode: "form" }

def _format_3di(seq: str) -> str:
    return "<fold2AA> " + " ".join(list(seq.lower()))


def _decode_aa(token_ids: torch.Tensor) -> str:
    s = tokenizer.decode(token_ids, skip_special_tokens=True)
    return "".join(s.split())


@torch.inference_mode()
def generate_reference(three_di: str) -> str:
    """Plain enc-dec greedy generation (the ground truth for correctness checks)."""
    L = len(three_di)
    enc = tokenizer([_format_3di(three_di)], add_special_tokens=True, return_tensors="pt").to(device)
    out = model.generate(
        input_ids=enc.input_ids,
        attention_mask=enc.attention_mask,
        max_length=L + 2,
        do_sample=False,
        num_beams=1,
    )
    return _decode_aa(out[0])[:L]

In [8]:
def _trim_kv_cache(past_key_values, target_len: int):
    """Trim a HuggingFace T5 past_key_values to target_len decoder positions.

    T5's past has 4 tensors per layer:
      (self_attn_K, self_attn_V, cross_attn_K, cross_attn_V)
    Only self_attn_K and self_attn_V grow with decoder length and need trimming;
    the cross-attn tensors are keyed on encoder positions and stay constant.
    Self-attn tensors have shape (batch, n_heads, decoder_len, head_dim).
    """
    trimmed = []
    for layer_past in past_key_values:
        sa_k, sa_v, ca_k, ca_v = layer_past
        trimmed.append((
            sa_k[:, :, :target_len, :],
            sa_v[:, :, :target_len, :],
            ca_k,
            ca_v,
        ))
    return tuple(trimmed)

In [9]:
#@title Core: speculative_decode() — greedy mode. { display-mode: "form" }

@torch.inference_mode()
def speculative_decode_greedy(three_di: str, K: int = 5,
                               get_draft_logits=None,
                               dynamic_k: bool = False,
                               K_init: int = 5,
                               K_min: int = 1,
                               K_max: int = 16) -> dict:
    """
    Speculative decoding with greedy verification, using KV-cache.
    Output is EXACTLY the same as plain enc-dec greedy generation.
    """
    L = len(three_di)

    # 1. Encode (shared, one-time)
    enc_input = tokenizer(
        [_format_3di(three_di)], add_special_tokens=True, return_tensors="pt"
    ).to(device)
    encoder_out = encoder(
        input_ids=enc_input.input_ids,
        attention_mask=enc_input.attention_mask,
    )
    encoder_hidden = encoder_out.last_hidden_state  # (1, seq_len, 1024)
    encoder_attention_mask = enc_input.attention_mask

    # Default to CNN drafter (prefix-independent)
    if get_draft_logits is None:
        h_cnn = encoder_hidden[:, 1:-1, :]
        cnn_logits_all = cnn(h_cnn.float())[0]  # (L, 20)
        def get_draft_logits(pos, accepted_prefix, enc_hidden):
            k = min(K, L - pos)
            return cnn_logits_all[pos:pos+k]  # (k, 20) — ignores prefix

    # Speculative decoding loop state
    generated_token_ids = [DECODER_START_TOKEN_ID]
    accepted_counts = []
    k_history = []
    k_current = K_init if dynamic_k else K
    n_steps = 0

    # KV-cache state
    past_key_values = None
    # cached_len = number of decoder positions currently represented in the cache.
    # Invariant: cached_len <= len(generated_token_ids).
    # The "uncached suffix" to feed next is generated_token_ids[cached_len:] + drafts.
    cached_len = 0

    if device.type == "cuda":
        torch.cuda.synchronize()
    t_start = time.perf_counter()

    while len(generated_token_ids) - 1 < L:
        pos = len(generated_token_ids) - 1
        k = min(k_current, L - pos)
        if k <= 0:
            break
        k_history.append(k)

        # Draft: k logits from drafter
        draft_logits_20 = get_draft_logits(pos, generated_token_ids[1:], encoder_hidden)
        k = min(k, draft_logits_20.shape[0])
        draft_cnn_indices = draft_logits_20[:k].argmax(dim=-1)
        draft_token_ids = [CNN_IDX_TO_TOKEN_ID[int(idx)] for idx in draft_cnn_indices]

        # Build verify input: only the uncached suffix.
        # First iteration: cached_len=0, suffix = [BOS] + drafts.
        # Later iterations: cached_len > 0, suffix = newly-confirmed tokens since
        # last call + drafts (in steady state that's just the bonus token + drafts).
        suffix_ids = generated_token_ids[cached_len:] + draft_token_ids
        verify_input = torch.tensor([suffix_ids], device=device, dtype=torch.long)

        dec_out = model(
            encoder_outputs=(encoder_hidden,),
            attention_mask=encoder_attention_mask,
            decoder_input_ids=verify_input,
            past_key_values=past_key_values,
            use_cache=True,
        )
        # verify_logits has shape (suffix_len, vocab). Position i in this tensor
        # corresponds to absolute decoder position (cached_len + i), and predicts
        # the token at absolute position (cached_len + i + 1).
        verify_logits = dec_out.logits[0]
        new_past = dec_out.past_key_values
        # After this call, the cache covers positions [0, cached_len + suffix_len).
        new_cached_len = cached_len + verify_input.shape[1]

        # We need verifier predictions for the k draft positions.
        # Draft i sits at absolute decoder position (pos + 1 + i) in generated_token_ids
        # (i.e. it would become generated_token_ids[pos + 1 + i] if accepted).
        # The verifier prediction for that token comes from the logits AT absolute
        # position (pos + i), which is local index (pos + i - cached_len) in verify_logits.
        base_local = pos - cached_len  # local index of logits that predict draft[0]

        # Vor der Schleife: alle Verifier-Predictions auf einmal holen
        verifier_tokens = verify_logits[base_local : base_local + k + 1].argmax(dim=-1).tolist()
        # Liste der Länge k+1: [pred_for_draft_0, pred_for_draft_1, ..., bonus_token]
        
        n_accepted = 0
        rejected_replacement = None
        for i in range(k):
            if draft_token_ids[i] == verifier_tokens[i]:
                n_accepted += 1
            else:
                rejected_replacement = verifier_tokens[i]
                break

        if rejected_replacement is None:
            # All k drafts accepted — also take the bonus token from the trailing logit.
            generated_token_ids.extend(draft_token_ids)
            bonus_local_idx = base_local + k  # logits AT position (pos + k) predict token (pos + k + 1)
            if len(generated_token_ids) - 1 < L and bonus_local_idx < verify_logits.shape[0]:
                bonus_token = verify_logits[bonus_local_idx].argmax(dim=-1).item()
                generated_token_ids.append(bonus_token)
            # Cache now correctly covers all of generated_token_ids (the bonus token
            # is predicted by an existing cache slot but is not itself in the cache yet —
            # it will be the "uncached suffix" on the next iteration).
            past_key_values = new_past
            cached_len = new_cached_len
        else:
            # Partial accept: keep n_accepted drafts, append the verifier correction.
            generated_token_ids.extend(draft_token_ids[:n_accepted])
            generated_token_ids.append(rejected_replacement)
            # The cache currently covers (cached_len + suffix_len) positions, but
            # only the first (cached_len + 1 + n_accepted) of those correspond to
            # tokens we actually committed to (BOS through the last accepted draft).
            # The verifier's correction token is NOT yet in the cache. Trim back.
            valid_cache_len = cached_len + 1 + n_accepted  # +1 for the token BEFORE drafts
            # Actually: suffix was [generated_token_ids[cached_len:pos+1]] + drafts
            #   = (pos + 1 - cached_len) old tokens + k drafts.
            # Of those, we keep (pos + 1 - cached_len) + n_accepted positions in the cache.
            valid_cache_len = cached_len + (pos + 1 - cached_len) + n_accepted
            #                = pos + 1 + n_accepted
            past_key_values = _trim_kv_cache(new_past, valid_cache_len)
            cached_len = valid_cache_len

        accepted_counts.append(n_accepted)
        n_steps += 1

        if dynamic_k:
            if n_accepted == k:
                k_current = min(k_current + 2, K_max)
            else:
                k_current = max(k_current - 1, K_min)

    if device.type == "cuda":
        torch.cuda.synchronize()
    wall_time = time.perf_counter() - t_start

    output_ids = generated_token_ids[1:]
    output_aa = _decode_aa(torch.tensor(output_ids))[:L]
    tokens_per_step = [(acc + 1) for acc in accepted_counts]

    return {
        "sequence": output_aa,
        "wall_time": wall_time,
        "n_steps": n_steps,
        "n_tokens": len(output_aa),
        "K": K if not dynamic_k else K_init,
        "dynamic_k": dynamic_k,
        "k_history": k_history,
        "accepted_counts": accepted_counts,
        "mean_accepted": np.mean(accepted_counts) if accepted_counts else 0,
        "tokens_per_step": tokens_per_step,
        "mean_tokens_per_step": np.mean(tokens_per_step) if tokens_per_step else 0,
        "acceptance_rate": np.mean(accepted_counts) / np.mean(k_history) if k_history else 0,
    }

In [10]:
#@title Core: speculative_decode() — stochastic mode. { display-mode: "form" }

@torch.inference_mode()
def speculative_decode_stochastic(three_di: str, K: int = 5,
                                   temperature: float = 1.0) -> dict:
    """
    Speculative decoding with Leviathan's stochastic accept/reject rule.
    Samples from EXACTLY the same distribution as enc-dec with sampling.
    
    Accept token x with probability min(1, p(x)/q(x)).
    On rejection: resample from norm(max(0, p - q)).
    """
    L = len(three_di)
    
    # 1. Encode
    enc_input = tokenizer(
        [_format_3di(three_di)], add_special_tokens=True, return_tensors="pt"
    ).to(device)
    encoder_out = encoder(
        input_ids=enc_input.input_ids,
        attention_mask=enc_input.attention_mask,
    )
    encoder_hidden = encoder_out.last_hidden_state
    
    # 2. CNN logits
    h_cnn = encoder_hidden[:, 1:-1, :]
    cnn_logits_20 = cnn(h_cnn.float())[0]  # (L_cnn, 20)
    L_cnn = cnn_logits_20.shape[0]
    
    # 3. Speculative decoding loop
    generated_token_ids = [DECODER_START_TOKEN_ID]
    accepted_counts = []
    n_steps = 0
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    t_start = time.perf_counter()
    
    while len(generated_token_ids) - 1 < L:
        pos = len(generated_token_ids) - 1
        k = min(K, L - pos, L_cnn - pos)
        if k <= 0:
            break
        
        # Draft: sample from CNN distribution
        q_logits = cnn_logits_20[pos:pos+k] / max(temperature, 1e-8)  # (k, 20)
        q_probs = F.softmax(q_logits, dim=-1)  # (k, 20)
        draft_cnn_indices = torch.multinomial(q_probs, num_samples=1).squeeze(-1)  # (k,)
        draft_token_ids = [CNN_IDX_TO_TOKEN_ID[int(idx)] for idx in draft_cnn_indices]
        
        # Verify
        verify_input = torch.tensor(
            [generated_token_ids + draft_token_ids], device=device, dtype=torch.long
        )
        dec_out = model(
            encoder_outputs=(encoder_hidden,),
            decoder_input_ids=verify_input,
        )
        verify_logits = dec_out.logits[0]
        
        # Accept/reject with Leviathan's rule
        n_accepted = 0
        for i in range(k):
            logit_idx = pos + i
            
            # Verifier distribution (over full vocab, but we only care about AA tokens)
            p_logits_full = verify_logits[logit_idx] / max(temperature, 1e-8)
            p_probs_full = F.softmax(p_logits_full, dim=-1)
            
            # Get p and q for the drafted token
            drafted_vocab_id = draft_token_ids[i]
            drafted_cnn_idx = int(draft_cnn_indices[i])
            
            p_x = p_probs_full[drafted_vocab_id].item()
            q_x = q_probs[i, drafted_cnn_idx].item()
            
            # Accept with probability min(1, p(x)/q(x))
            accept_prob = min(1.0, p_x / max(q_x, 1e-10))
            r = torch.rand(1).item()
            
            if r < accept_prob:
                n_accepted += 1
            else:
                # Reject: resample from max(0, p - q) restricted to AA tokens
                p_aa = p_probs_full[AA_TOKEN_IDS]  # (20,)
                q_aa = q_probs[i]  # (20,)
                adjusted = torch.clamp(p_aa - q_aa, min=0)
                adjusted_sum = adjusted.sum()
                if adjusted_sum > 1e-10:
                    adjusted = adjusted / adjusted_sum
                    resampled_cnn_idx = torch.multinomial(adjusted, num_samples=1).item()
                else:
                    resampled_cnn_idx = torch.multinomial(p_aa, num_samples=1).item()
                resampled_token_id = AA_TOKEN_IDS[resampled_cnn_idx]
                
                generated_token_ids.extend(draft_token_ids[:n_accepted])
                generated_token_ids.append(resampled_token_id)
                break
        else:
            # All accepted — sample bonus token from verifier
            generated_token_ids.extend(draft_token_ids)
            if len(generated_token_ids) - 1 < L:
                bonus_logit_idx = pos + k
                if bonus_logit_idx < verify_logits.shape[0]:
                    bonus_logits = verify_logits[bonus_logit_idx] / max(temperature, 1e-8)
                    bonus_probs = F.softmax(bonus_logits, dim=-1)
                    bonus_probs_aa = bonus_probs[AA_TOKEN_IDS]
                    bonus_probs_aa = bonus_probs_aa / bonus_probs_aa.sum()
                    bonus_idx = torch.multinomial(bonus_probs_aa, num_samples=1).item()
                    generated_token_ids.append(AA_TOKEN_IDS[bonus_idx])
        
        accepted_counts.append(n_accepted)
        n_steps += 1
    
    if device.type == "cuda":
        torch.cuda.synchronize()
    wall_time = time.perf_counter() - t_start
    
    output_ids = generated_token_ids[1:]
    output_aa = _decode_aa(torch.tensor(output_ids))[:L]
    
    return {
        "sequence": output_aa,
        "wall_time": wall_time,
        "n_steps": n_steps,
        "n_tokens": len(output_aa),
        "K": K,
        "temperature": temperature,
        "accepted_counts": accepted_counts,
        "mean_accepted": np.mean(accepted_counts) if accepted_counts else 0,
        "acceptance_rate": np.mean(accepted_counts) / K if accepted_counts else 0,
    }

In [11]:
#@title Correctness verification: greedy spec-decode == enc-dec greedy. { display-mode: "form" }

print("Verifying correctness: greedy speculative decoding must produce")
print("EXACTLY the same output as plain enc-dec greedy generation.\n")

# Test on first 5 shortest proteins
test_proteins = sorted(test_set.items(), key=lambda kv: kv[1]["length"])[:5]

all_correct = True
for uid, rec in test_proteins:
    ref = generate_reference(rec["3di"])
    for K in [3, 5, 8]:
        result = speculative_decode_greedy(rec["3di"], K=K)
        match = result["sequence"] == ref
        status = "PASS" if match else "FAIL"
        print(f"  {status}  {uid} L={rec['length']}  K={K}  "
              f"steps={result['n_steps']}  accept_rate={result['acceptance_rate']:.2f}")
        if not match:
            all_correct = False
            # Show first mismatch
            for j, (a, b) in enumerate(zip(ref, result["sequence"])):
                if a != b:
                    print(f"    First mismatch at pos {j}: ref={a} spec={b}")
                    break

print(f"\n{'ALL CORRECT' if all_correct else 'FAILURES DETECTED'}")
if not all_correct:
    print("WARNING: Bug in speculative decoding implementation!")

Verifying correctness: greedy speculative decoding must produce
EXACTLY the same output as plain enc-dec greedy generation.



Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


  PASS  P01542 L=46  K=3  steps=29  accept_rate=0.21
  PASS  P01542 L=46  K=5  steps=29  accept_rate=0.13
  PASS  P01542 L=46  K=8  steps=29  accept_rate=0.09
  PASS  P0A7N4 L=57  K=3  steps=41  accept_rate=0.13
  PASS  P0A7N4 L=57  K=5  steps=41  accept_rate=0.08
  PASS  P0A7N4 L=57  K=8  steps=41  accept_rate=0.05
  PASS  P02798 L=61  K=3  steps=49  accept_rate=0.08
  PASS  P02798 L=61  K=5  steps=49  accept_rate=0.05
  PASS  P02798 L=61  K=8  steps=49  accept_rate=0.03
  PASS  P61583 L=75  K=3  steps=36  accept_rate=0.36
  PASS  P61583 L=75  K=5  steps=35  accept_rate=0.23
  PASS  P61583 L=75  K=8  steps=35  accept_rate=0.15
  PASS  P24311 L=80  K=3  steps=46  accept_rate=0.26
  PASS  P24311 L=80  K=5  steps=44  accept_rate=0.17
  PASS  P24311 L=80  K=8  steps=44  accept_rate=0.11

ALL CORRECT


## Part 3 — HuggingFace `assistant_model` Wrapper

Wraps the CNN as a compatible `PreTrainedModel` so we can use `model.generate(..., assistant_model=cnn_wrapper)`.

In [12]:
#@title CNNAssistantModel wrapper. { display-mode: "form" }
from transformers import PreTrainedModel, T5Config
from transformers.generation.utils import GenerateDecoderOnlyOutput


class CNNAssistantModel(PreTrainedModel):
    """Wraps enc-CNN as a HuggingFace-compatible assistant model for speculative decoding.
    
    The CNN is prefix-independent: it always produces the same logits for a given
    encoder input regardless of what was previously generated. We exploit this by
    pre-computing all logits once and serving slices on demand.
    """
    config_class = T5Config
    
    def __init__(self, config, prostt5_encoder, cnn_head, tok, aa_token_ids):
        super().__init__(config)
        self._encoder = prostt5_encoder
        self._cnn = cnn_head
        self._tokenizer = tok
        self._aa_token_ids = aa_token_ids
        self.config.is_encoder_decoder = True
        self.config.decoder_start_token_id = config.decoder_start_token_id
        self.generation_config = GenerationConfig(
            num_assistant_tokens=5,
            num_assistant_tokens_schedule="constant",
            do_sample=False,
            max_length=3000,
            return_dict_in_generate=True,
            output_scores=True,
        )
        self._cached_logits = None
        self._cached_input_hash = None
    
    def _compute_cnn_logits(self, encoder_outputs):
        """Run CNN on encoder hidden states, return full-vocab logits for all positions."""
        hidden = encoder_outputs[0]  # (1, seq_len, 1024)
        h = hidden[:, 1:-1, :]  # trim prefix + EOS
        logits_20 = self._cnn(h.float())[0]  # (L, 20)
        # Map to full vocab: set all non-AA positions to -inf
        vocab_size = self.config.vocab_size
        full_logits = torch.full((logits_20.shape[0], vocab_size), -100.0,
                                 device=logits_20.device)
        for i, tid in enumerate(self._aa_token_ids):
            full_logits[:, tid] = logits_20[:, i]
        return full_logits
    
    def get_encoder(self):
        return self._encoder
    
    def prepare_inputs_for_generation(self, decoder_input_ids, encoder_outputs=None, **kwargs):
        return {
            "decoder_input_ids": decoder_input_ids,
            "encoder_outputs": encoder_outputs,
        }
    
    def forward(self, decoder_input_ids=None, encoder_outputs=None, **kwargs):
        """Return pre-computed CNN logits for the positions being queried."""
        if encoder_outputs is not None:
            # Compute or use cached logits
            hidden_id = id(encoder_outputs[0])
            if self._cached_input_hash != hidden_id:
                self._cached_logits = self._compute_cnn_logits(encoder_outputs)
                self._cached_input_hash = hidden_id
        
        # decoder_input_ids shape: (1, seq_len)
        seq_len = decoder_input_ids.shape[1]
        # We need to return logits for each position in decoder_input_ids
        # Position i predicts position i+1, so we return logits for positions 0..seq_len-1
        # which predict AA positions 0..seq_len-1 (after decoder_start)
        n_positions = seq_len
        if self._cached_logits is not None:
            L_cached = self._cached_logits.shape[0]
            n_to_return = min(n_positions, L_cached)
            logits = self._cached_logits[:n_to_return].unsqueeze(0)  # (1, n, vocab)
            # Pad if needed
            if n_to_return < n_positions:
                pad = torch.full((1, n_positions - n_to_return, logits.shape[-1]),
                                -100.0, device=logits.device)
                logits = torch.cat([logits, pad], dim=1)
        else:
            logits = torch.zeros((1, n_positions, self.config.vocab_size), device=device)
        
        from transformers.modeling_outputs import Seq2SeqLMOutput
        return Seq2SeqLMOutput(logits=logits)


# Instantiate
assistant_config = model.config
cnn_assistant = CNNAssistantModel(
    config=assistant_config,
    prostt5_encoder=encoder,
    cnn_head=cnn,
    tok=tokenizer,
    aa_token_ids=AA_TOKEN_IDS,
).to(device).eval()

print(f"CNN assistant model created. is_encoder_decoder={cnn_assistant.config.is_encoder_decoder}")
print(f"num_assistant_tokens={cnn_assistant.generation_config.num_assistant_tokens}")

CNNAssistantModel has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


CNN assistant model created. is_encoder_decoder=True
num_assistant_tokens=5


In [13]:
#@title Test HF assisted generation on a single protein. { display-mode: "form" }

# Pick shortest protein for quick test
test_uid, test_rec = sorted(test_set.items(), key=lambda kv: kv[1]["length"])[0]
print(f"Testing HF assisted generation on {test_uid} (L={test_rec['length']})")

enc_input = tokenizer(
    [_format_3di(test_rec["3di"])], add_special_tokens=True, return_tensors="pt"
).to(device)

# Reference (plain greedy)
ref = generate_reference(test_rec["3di"])
print(f"Reference (plain enc-dec): {ref[:50]}...")

# Try HF assisted generation
try:
    hf_out = model.generate(
        input_ids=enc_input.input_ids,
        attention_mask=enc_input.attention_mask,
        max_length=test_rec["length"] + 2,
        do_sample=False,
        num_beams=1,
        assistant_model=cnn_assistant,
    )
    hf_result = _decode_aa(hf_out[0])[:test_rec["length"]]
    match = hf_result == ref
    print(f"HF assisted result:       {hf_result[:50]}...")
    print(f"Match with reference: {match}")
    if not match:
        print("NOTE: HF wrapper may need adjustments. Custom implementation is primary.")
except Exception as e:
    print(f"HF assisted generation failed: {e}")
    print("This is expected if the wrapper needs further adaptation.")
    print("The custom implementation (Part 1) is the primary approach.")

Testing HF assisted generation on P01542 (L=46)
Reference (plain enc-dec): MYCCPSAAARNALMAQVGTGVPLDQAAAAAGCTVVTSGTCPAGQAP...
HF assisted generation failed: The following `model_kwargs` are not used by the model: ['attention_mask'] (note: typos in the generate arguments will also show up in this list)
This is expected if the wrapper needs further adaptation.
The custom implementation (Part 1) is the primary approach.


In [14]:
#@title Timing helpers. { display-mode: "form" }

def _sync():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def _reset_peak_mem():
    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)


def _peak_mem_gb() -> float:
    if device.type == "cuda":
        return torch.cuda.max_memory_allocated(device) / 1e9
    return 0.0


@torch.inference_mode()
def time_encdec(three_di: str) -> float:
    """Time plain enc-dec greedy generation. Returns median wall time."""
    L = len(three_di)
    enc = tokenizer([_format_3di(three_di)], add_special_tokens=True, return_tensors="pt").to(device)
    gen_kwargs = dict(
        input_ids=enc.input_ids, attention_mask=enc.attention_mask,
        max_length=L+2, do_sample=False, num_beams=1,
    )
    for _ in range(NUM_WARMUP):
        model.generate(**gen_kwargs)
    _sync()
    times = []
    for _ in range(NUM_REPEATS):
        _sync()
        t0 = time.perf_counter()
        model.generate(**gen_kwargs)
        _sync()
        times.append(time.perf_counter() - t0)
    return statistics.median(times)


def time_spec_decode(three_di: str, K: int,
                     get_draft_logits=None,
                     dynamic_k: bool = False,
                     K_init: int = 5, K_min: int = 1, K_max: int = 16) -> tuple[float, dict]:
    """Time speculative decoding with any drafter. Returns (median_wall_time, last_result_dict)."""
    kwargs = dict(K=K, get_draft_logits=get_draft_logits,
                  dynamic_k=dynamic_k, K_init=K_init, K_min=K_min, K_max=K_max)
    for _ in range(NUM_WARMUP):
        speculative_decode_greedy(three_di, **kwargs)
    _sync()
    times = []
    last_result = None
    for _ in range(NUM_REPEATS):
        result = speculative_decode_greedy(three_di, **kwargs)
        times.append(result["wall_time"])
        last_result = result
    return statistics.median(times), last_result


print("Timing helpers defined.")

Timing helpers defined.


## Part 4 — Benchmarking

Measure real speedup across all proteins and K values. Compare to baseline enc-dec timing and α-predicted speedup.

In [15]:
#@title Checkpointing. { display-mode: "form" }

def save_spec_checkpoint(state: dict):
    state["timestamp"] = datetime.now().isoformat()
    path = CHECKPOINT_DIR / "spec_decode_checkpoint.pkl"
    with open(path, "wb") as f:
        pickle.dump(state, f)


def load_spec_checkpoint() -> dict | None:
    path = CHECKPOINT_DIR / "spec_decode_checkpoint.pkl"
    if path.exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    return None


print(f"Spec-decode checkpoint dir: {CHECKPOINT_DIR}")

Spec-decode checkpoint dir: /kaggle/working/prostT5_benchmarks/spec_decode_checkpoints


In [ ]:
#@title Main benchmark loop — CNN + HMM drafters. { display-mode: "form" }

# ── Build HMM drafters (fetches Pfam MSAs, ~2-5 min for 100 proteins) ─────────
# print("Building HMM drafters for test set...")
# print("Proteins without a Pfam family will fall back to CNN drafter.\n")
# hmm_drafters = build_hmm_drafters_for_test_set(test_set)
# n_hmm = sum(1 for v in hmm_drafters.values() if v is not None)
# print(f"\nHMM coverage: {n_hmm}/{len(test_set)} proteins")

# ── Load or initialize checkpoint ─────────────────────────────────────────────
ckpt = load_spec_checkpoint()
if ckpt is not None:
    results = ckpt["results"]
    completed = set(ckpt["completed"])
    print(f"Resuming: {len(completed)} tasks done.")
else:
    results = []
    completed = set()

# ── Drafters to benchmark ──────────────────────────────────────────────────────
DRAFTERS = [
    ("cnn", None)   # enc-CNN, prefix-independent
    # ("hmm", "hmm"),  # Profile HMM, prefix-aware (looked up per protein)
]

# K=-1 is the sentinel for dynamic K rows (avoids mixed int/str column type)
K_DYNAMIC = -1

sorted_proteins = sorted(test_set.items(), key=lambda kv: kv[1]["length"])
n_dynamic = len(DRAFTERS)
total_tasks = len(sorted_proteins) * (1 + len(DRAFTERS) * len(K_VALUES) + n_dynamic)
print(f"Total tasks: {len(sorted_proteins)} proteins × "
      f"({len(DRAFTERS)} drafters × {len(K_VALUES)} K + {n_dynamic} dynamic + 1 enc-dec) = {total_tasks}")
print(f"Already done: {len(completed)}, Remaining: {total_tasks - len(completed)}")

t_global_start = time.time()

for uid, rec in sorted_proteins:
    L = rec["length"]
    three_di = rec["3di"]

    # ── enc-dec baseline (once per protein) ───────────────────────────────────
    encdec_key = f"{uid}_encdec"
    if encdec_key not in completed:
        t_enc = time_encdec(three_di)
        results.append({
            "protein_id": uid, "length": L, "drafter": "enc_dec",
            "K": 0, "wall_s": t_enc, "speedup": 1.0,
            "acceptance_rate": None, "mean_tokens_per_step": 1.0, "n_steps": L,
        })
        completed.add(encdec_key)
        print(f"\n[{uid} L={L}]  enc-dec: {t_enc:.2f}s")
    else:
        t_enc = next((r["wall_s"] for r in results
                      if r["protein_id"] == uid and r["drafter"] == "enc_dec"), None)

    # ── Spec-decode: each drafter × each K + one dynamic run ──────────────────
    for drafter_name, drafter_fn in DRAFTERS:
        # Resolve HMM drafter for this protein
        if drafter_fn == "hmm":
            drafter_obj = hmm_drafters.get(uid)
            if drafter_obj is None:
                continue
            fn = drafter_obj.get_draft_logits
        else:
            fn = None

        for K in K_VALUES:
            task_key = f"{uid}_{drafter_name}_K{K}"
            if task_key in completed:
                continue

            t_spec, spec_result = time_spec_decode(three_di, K=K, get_draft_logits=fn)
            speedup = t_enc / t_spec if t_spec > 0 else float("nan")
            results.append({
                "protein_id": uid, "length": L, "drafter": drafter_name,
                "K": K, "wall_s": t_spec, "speedup": speedup,
                "acceptance_rate": spec_result["acceptance_rate"],
                "mean_tokens_per_step": spec_result["mean_tokens_per_step"],
                "n_steps": spec_result["n_steps"],
            })
            completed.add(task_key)
            print(f"  {drafter_name} K={K:2d}: {t_spec:.2f}s  "
                  f"speedup={speedup:.2f}x  accept={spec_result['acceptance_rate']:.2f}")

        # ── Dynamic K run (one per drafter, K_DYNAMIC=-1 sentinel) ────────────
        dyn_task_key = f"{uid}_{drafter_name}_dynamic"
        if dyn_task_key not in completed:
            t_dyn, dyn_result = time_spec_decode(
                three_di, K=5, get_draft_logits=fn,
                dynamic_k=True, K_init=5, K_min=1, K_max=16,
            )
            speedup_dyn = t_enc / t_dyn if t_dyn > 0 else float("nan")
            results.append({
                "protein_id": uid, "length": L, "drafter": drafter_name,
                "K": K_DYNAMIC, "wall_s": t_dyn, "speedup": speedup_dyn,
                "acceptance_rate": dyn_result["acceptance_rate"],
                "mean_tokens_per_step": dyn_result["mean_tokens_per_step"],
                "n_steps": dyn_result["n_steps"],
                "k_history": dyn_result["k_history"],
            })
            completed.add(dyn_task_key)
            mean_k = np.mean(dyn_result["k_history"]) if dyn_result["k_history"] else 0
            print(f"  {drafter_name} K=dyn: {t_dyn:.2f}s  "
                  f"speedup={speedup_dyn:.2f}x  accept={dyn_result['acceptance_rate']:.2f}  "
                  f"mean_K={mean_k:.1f}")

    # Checkpoint after each protein
    save_spec_checkpoint({"results": results, "completed": list(completed)})

    elapsed = time.time() - t_global_start
    done = len(completed)
    remaining = total_tasks - done
    if done > 0:
        eta_min = (elapsed / done) * remaining / 60
        print(f"  ETA: ~{eta_min:.0f} min")

    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print(f"\n{'='*60}")
print(f"BENCHMARK COMPLETE")

Total tasks: 97 proteins × (1 drafters × 5 K + 1 dynamic + 1 enc-dec) = 679
Already done: 0, Remaining: 679

[P01542 L=46]  enc-dec: 2.06s
  cnn K= 1: 1.51s  speedup=1.36x  accept=0.42
  cnn K= 3: 1.31s  speedup=1.58x  accept=0.21
  cnn K= 5: 1.35s  speedup=1.52x  accept=0.13
  cnn K= 8: 1.36s  speedup=1.51x  accept=0.09
  cnn K=16: 1.35s  speedup=1.52x  accept=0.05
  cnn K=dyn: 1.44s  speedup=1.43x  accept=0.26  mean_K=2.0
  ETA: ~85 min

[P0A7N4 L=57]  enc-dec: 2.59s
  cnn K= 1: 2.11s  speedup=1.23x  accept=0.27
  cnn K= 3: 1.89s  speedup=1.37x  accept=0.13
  cnn K= 5: 1.92s  speedup=1.35x  accept=0.08
  cnn K= 8: 1.88s  speedup=1.38x  accept=0.05
  cnn K=16: 1.94s  speedup=1.34x  accept=0.03
  cnn K=dyn: 2.09s  speedup=1.24x  accept=0.16  mean_K=1.6
  ETA: ~100 min

[P02798 L=61]  enc-dec: 2.71s
  cnn K= 1: 2.29s  speedup=1.19x  accept=0.22
  cnn K= 3: 2.21s  speedup=1.23x  accept=0.08


## Part 5 — Analysis & Plots

In [ ]:
#@title Aggregate results. { display-mode: "form" }
import pandas as pd

df = pd.DataFrame(results)
df.to_csv(RESULTS_DIR / "spec_decode_results.csv", index=False)

encdec_df = df[df["drafter"] == "enc_dec"]
spec_df    = df[df["drafter"] != "enc_dec"]
fixed_df   = spec_df[spec_df["K"] != K_DYNAMIC]   # K in {1,3,5,8,16}
dyn_df     = spec_df[spec_df["K"] == K_DYNAMIC]    # dynamic K runs

print("=== Speedup by drafter and K ===")
print(f"{'Drafter':>8s}  {'K':>7s}  {'Mean Speedup':>12s}  {'Median':>8s}  {'Accept':>8s}  {'Tok/step':>8s}")
print("-" * 65)
for drafter in ["cnn", "hmm"]:
    for K in K_VALUES:
        subset = fixed_df[(fixed_df["drafter"] == drafter) & (fixed_df["K"] == K)]
        if len(subset) > 0:
            print(f"{drafter:>8s}  {K:7d}  "
                  f"{subset['speedup'].mean():12.2f}  "
                  f"{subset['speedup'].median():8.2f}  "
                  f"{subset['acceptance_rate'].mean():8.3f}  "
                  f"{subset['mean_tokens_per_step'].mean():8.2f}")
    # Dynamic row
    dyn_sub = dyn_df[dyn_df["drafter"] == drafter]
    if len(dyn_sub) > 0:
        mean_k_vals = dyn_sub["k_history"].apply(lambda h: np.mean(h) if isinstance(h, list) and h else float("nan"))
        print(f"{drafter:>8s}  {'dynamic':>7s}  "
              f"{dyn_sub['speedup'].mean():12.2f}  "
              f"{dyn_sub['speedup'].median():8.2f}  "
              f"{dyn_sub['acceptance_rate'].mean():8.3f}  "
              f"{dyn_sub['mean_tokens_per_step'].mean():8.2f}  "
              f"(mean_K={mean_k_vals.mean():.1f})")

print(f"\nCNN proteins (fixed K): {fixed_df[fixed_df['drafter']=='cnn']['protein_id'].nunique()}")
print(f"HMM proteins (fixed K): {fixed_df[fixed_df['drafter']=='hmm']['protein_id'].nunique()} "
      f"(only proteins with Pfam families)")
print(f"Dynamic K runs — CNN: {dyn_df[dyn_df['drafter']=='cnn']['protein_id'].nunique()}, "
      f"HMM: {dyn_df[dyn_df['drafter']=='hmm']['protein_id'].nunique()}")

In [ ]:
#@title Compare predicted (from α) vs. measured speedup. { display-mode: "form" }

if alpha_predictions:
    comparison_rows = []
    for uid in spec_df["protein_id"].unique():
        if uid not in alpha_predictions:
            continue
        alpha_data = alpha_predictions[uid]
        # Use greedy T=1.0 alpha
        greedy_alpha = alpha_data.get("greedy_T1.0", {}).get("alpha_mean", None)
        if greedy_alpha is None:
            continue
        
        for K in K_VALUES:
            measured = spec_df[(spec_df["protein_id"] == uid) & (spec_df["K"] == K)]
            if len(measured) == 0:
                continue
            
            # Predicted tokens per step from Theorem 3.8
            a = greedy_alpha
            if a < 1.0:
                predicted_tok_per_step = (1 - a**(K+1)) / (1 - a)
            else:
                predicted_tok_per_step = K + 1
            
            comparison_rows.append({
                "protein_id": uid,
                "K": K,
                "alpha": a,
                "predicted_tok_per_step": predicted_tok_per_step,
                "measured_tok_per_step": measured["mean_tokens_per_step"].values[0],
                "measured_speedup": measured["speedup"].values[0],
                "measured_acceptance": measured["acceptance_rate"].values[0],
            })
    
    comp_df = pd.DataFrame(comparison_rows)
    comp_df.to_csv(RESULTS_DIR / "predicted_vs_measured.csv", index=False)
    
    print("=== Predicted vs. Measured (tokens/step) ===")
    print(f"{'K':>3s}  {'Predicted':>10s}  {'Measured':>10s}  {'Ratio':>7s}")
    print("-" * 35)
    for K in K_VALUES:
        subset = comp_df[comp_df["K"] == K]
        if len(subset) > 0:
            pred = subset["predicted_tok_per_step"].mean()
            meas = subset["measured_tok_per_step"].mean()
            print(f"{K:3d}  {pred:10.2f}  {meas:10.2f}  {meas/pred:7.2f}")
    
    print(f"\nIf ratio < 1.0: CNN's prefix-independence hurts acceptance in practice.")
    print(f"If ratio ~ 1.0: α prediction is accurate.")
else:
    print("No alpha predictions available. Run baseline notebook first.")
    comp_df = pd.DataFrame()

In [ ]:
#@title Plots: CNN vs HMM drafter comparison. { display-mode: "form" }
import matplotlib.pyplot as plt

COLORS = {"cnn": "steelblue", "hmm": "darkorange"}
MARKERS_DYN = {"cnn": "s", "hmm": "D"}
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# (0,0) Speedup vs K — CNN vs HMM, fixed + dynamic
ax = axes[0, 0]
for drafter in ["cnn", "hmm"]:
    sub = fixed_df[fixed_df["drafter"] == drafter]
    by_k = sub.groupby("K")["speedup"].median()
    ax.plot(by_k.index, by_k.values, "o-", label=drafter, color=COLORS[drafter])
    # Dynamic K point plotted at x = mean_K for that drafter
    dyn_sub = dyn_df[dyn_df["drafter"] == drafter]
    if len(dyn_sub) > 0:
        dyn_mean_k = dyn_sub["k_history"].apply(lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")).mean()
        dyn_speedup = dyn_sub["speedup"].median()
        ax.scatter([dyn_mean_k], [dyn_speedup], marker=MARKERS_DYN[drafter],
                   color=COLORS[drafter], s=80, zorder=5,
                   label=f"{drafter} dynamic (mean_K={dyn_mean_k:.1f})")
ax.axhline(1.0, color="red", linestyle="--", alpha=0.4, label="breakeven")
ax.set_xlabel("Draft length K")
ax.set_ylabel("Median speedup over enc-dec")
ax.set_title("Speedup vs. K")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (0,1) Acceptance rate vs K — CNN vs HMM, fixed + dynamic
ax = axes[0, 1]
for drafter in ["cnn", "hmm"]:
    sub = fixed_df[fixed_df["drafter"] == drafter]
    by_k = sub.groupby("K")["acceptance_rate"].mean()
    ax.plot(by_k.index, by_k.values, "o-", label=drafter, color=COLORS[drafter])
    dyn_sub = dyn_df[dyn_df["drafter"] == drafter]
    if len(dyn_sub) > 0:
        dyn_mean_k = dyn_sub["k_history"].apply(lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")).mean()
        ax.scatter([dyn_mean_k], [dyn_sub["acceptance_rate"].mean()],
                   marker=MARKERS_DYN[drafter], color=COLORS[drafter], s=80, zorder=5,
                   label=f"{drafter} dynamic")
ax.set_xlabel("Draft length K")
ax.set_ylabel("Mean acceptance rate")
ax.set_title("Acceptance rate vs. K")
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# (0,2) CNN vs HMM acceptance rate scatter (shared proteins, K=5)
ax = axes[0, 2]
K_compare = 5
cnn_k5 = fixed_df[(fixed_df["drafter"] == "cnn") & (fixed_df["K"] == K_compare)].set_index("protein_id")
hmm_k5 = fixed_df[(fixed_df["drafter"] == "hmm") & (fixed_df["K"] == K_compare)].set_index("protein_id")
shared = cnn_k5.index.intersection(hmm_k5.index)
if len(shared) > 0:
    ax.scatter(cnn_k5.loc[shared, "acceptance_rate"], hmm_k5.loc[shared, "acceptance_rate"],
               alpha=0.6, s=20)
    lim = max(cnn_k5.loc[shared, "acceptance_rate"].max(),
              hmm_k5.loc[shared, "acceptance_rate"].max()) * 1.05
    ax.plot([0, lim], [0, lim], "r--", alpha=0.4, label="CNN == HMM")
    ax.set_xlabel(f"CNN acceptance rate (K={K_compare})")
    ax.set_ylabel(f"HMM acceptance rate (K={K_compare})")
    ax.set_title("CNN vs HMM acceptance (shared proteins)")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "No shared proteins yet", ha="center", va="center", transform=ax.transAxes)

# (1,0) Speedup vs length — best fixed K, both drafters
ax = axes[1, 0]
best_k = fixed_df.groupby("K")["speedup"].median().idxmax() if len(fixed_df) > 0 else 5
for drafter in ["cnn", "hmm"]:
    sub = fixed_df[(fixed_df["drafter"] == drafter) & (fixed_df["K"] == best_k)]
    ax.scatter(sub["length"], sub["speedup"], alpha=0.5, s=15,
               label=drafter, color=COLORS[drafter])
ax.axhline(1.0, color="red", linestyle="--", alpha=0.4)
ax.set_xscale("log")
ax.set_xlabel("Protein length")
ax.set_ylabel(f"Speedup (K={best_k})")
ax.set_title(f"Speedup vs. length (best fixed K={best_k})")
ax.legend()
ax.grid(True, alpha=0.3)

# (1,1) Latency: enc-dec vs CNN vs HMM (best fixed K)
ax = axes[1, 1]
ax.scatter(encdec_df["length"], encdec_df["wall_s"], alpha=0.5, s=15,
           label="enc-dec", color="gray")
for drafter in ["cnn", "hmm"]:
    sub = fixed_df[(fixed_df["drafter"] == drafter) & (fixed_df["K"] == best_k)]
    ax.scatter(sub["length"], sub["wall_s"], alpha=0.5, s=15,
               label=f"spec {drafter} K={best_k}", color=COLORS[drafter])
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Protein length")
ax.set_ylabel("Wall time (s)")
ax.set_title("Latency comparison")
ax.legend()
ax.grid(True, which="both", alpha=0.3)

# (1,2) Predicted vs measured (CNN only, needs alpha from notebook 1)
ax = axes[1, 2]
if len(comp_df) > 0:
    ax.scatter(comp_df["predicted_tok_per_step"], comp_df["measured_tok_per_step"],
               alpha=0.4, s=15)
    max_val = max(comp_df["predicted_tok_per_step"].max(), comp_df["measured_tok_per_step"].max())
    ax.plot([0, max_val], [0, max_val], "r--", alpha=0.5, label="perfect prediction")
    ax.set_xlabel("Predicted tokens/step (α, Theorem 3.8)")
    ax.set_ylabel("Measured tokens/step")
    ax.set_title("Predicted vs. measured (CNN)")
    ax.legend()
    ax.grid(True, alpha=0.3)
else:
    ax.text(0.5, 0.5, "Run baseline notebook first\nfor α predictions",
            ha="center", va="center", transform=ax.transAxes)

fig.suptitle("ProstT5 Speculative Decoding: CNN vs Profile HMM Drafter", fontsize=13)
fig.tight_layout()
fig.savefig(RESULTS_DIR / "spec_decode_plots.png", dpi=150)
plt.show()
print(f"Saved to {RESULTS_DIR / 'spec_decode_plots.png'}")

In [ ]:
#@title Overhead analysis. { display-mode: "form" }

print("=== Overhead Analysis ===")
print(f"\nDraft model: enc-CNN (prefix-independent, one-shot)")
print(f"CNN params: {sum(p.numel() for p in cnn.parameters()):,} (vs ProstT5: {sum(p.numel() for p in model.parameters()):,})")
print(f"CNN overhead: single forward pass, negligible vs decoder verification")

# Measure CNN-only time on a few proteins
print(f"\n--- CNN drafting time (one-shot, all L positions) ---")
for uid, rec in sorted(test_set.items(), key=lambda kv: kv[1]["length"])[:5]:
    enc_input = tokenizer([_format_3di(rec["3di"])], add_special_tokens=True, return_tensors="pt").to(device)
    _sync()
    t0 = time.perf_counter()
    with torch.inference_mode():
        h = encoder(input_ids=enc_input.input_ids, attention_mask=enc_input.attention_mask).last_hidden_state
        h = h[:, 1:-1, :]
        _ = cnn(h.float())
    _sync()
    cnn_time = time.perf_counter() - t0
    print(f"  {uid} L={rec['length']:4d}: CNN forward = {cnn_time*1000:.1f}ms")

print(f"\nNote: encoder forward is shared between CNN and decoder verification,")
print(f"so CNN overhead is only the CNN head itself (< 1ms for any length).")

# vRAM comparison
if device.type == "cuda":
    _reset_peak_mem()
    uid, rec = sorted(test_set.items(), key=lambda kv: kv[1]["length"])[-1]  # longest
    _ = speculative_decode_greedy(rec["3di"], K=5)
    spec_vram = _peak_mem_gb()
    print(f"\nPeak vRAM during spec decode (longest protein, K=5): {spec_vram:.2f} GB")

In [ ]:
#@title Dynamic K behavior analysis. { display-mode: "form" }

if len(dyn_df) == 0:
    print("No dynamic K results yet.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # (0) Distribution of mean_K across proteins, per drafter
    ax = axes[0]
    for drafter in ["cnn", "hmm"]:
        sub = dyn_df[dyn_df["drafter"] == drafter]
        if len(sub) == 0:
            continue
        mean_ks = sub["k_history"].apply(
            lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")
        ).dropna()
        ax.hist(mean_ks, bins=15, alpha=0.6, label=drafter, color=COLORS[drafter])
    ax.axvline(5, color="gray", linestyle="--", alpha=0.5, label="K_init=5")
    ax.set_xlabel("Mean K over sequence")
    ax.set_ylabel("Number of proteins")
    ax.set_title("Distribution of mean dynamic K")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # (1) K trace for 6 representative proteins (3 high-α, 3 low-α), CNN drafter
    ax = axes[1]
    cnn_dyn = dyn_df[dyn_df["drafter"] == "cnn"].copy()
    cnn_dyn["mean_k"] = cnn_dyn["k_history"].apply(
        lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")
    )
    cnn_dyn_sorted = cnn_dyn.dropna(subset=["mean_k"]).sort_values("mean_k")
    # Pick 3 lowest and 3 highest mean_K proteins
    representatives = pd.concat([cnn_dyn_sorted.head(3), cnn_dyn_sorted.tail(3)])
    cmap = plt.cm.RdYlGn
    for i, (_, row) in enumerate(representatives.iterrows()):
        hist = row["k_history"]
        if not isinstance(hist, list) or len(hist) == 0:
            continue
        color = cmap(i / max(len(representatives) - 1, 1))
        label = f"{row['protein_id']} (L={row['length']}, mean_K={row['mean_k']:.1f})"
        ax.plot(hist, alpha=0.8, linewidth=1, color=color, label=label)
    ax.axhline(5, color="gray", linestyle="--", alpha=0.4, label="K_init=5")
    ax.set_xlabel("Decoding step")
    ax.set_ylabel("K used")
    ax.set_title("K trace — 3 easiest & 3 hardest proteins (CNN)")
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(True, alpha=0.3)

    # (2) mean_K vs acceptance rate scatter — does high acceptance → high mean_K?
    ax = axes[2]
    for drafter in ["cnn", "hmm"]:
        sub = dyn_df[dyn_df["drafter"] == drafter].copy()
        if len(sub) == 0:
            continue
        sub["mean_k"] = sub["k_history"].apply(
            lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")
        )
        ax.scatter(sub["acceptance_rate"], sub["mean_k"],
                   alpha=0.5, s=15, label=drafter, color=COLORS[drafter])
    ax.axhline(5, color="gray", linestyle="--", alpha=0.4, label="K_init=5")
    ax.set_xlabel("Acceptance rate")
    ax.set_ylabel("Mean K over sequence")
    ax.set_title("Acceptance rate vs. mean dynamic K")
    ax.legend()
    ax.grid(True, alpha=0.3)

    fig.suptitle("Dynamic K Behavior Analysis", fontsize=13)
    fig.tight_layout()
    fig.savefig(RESULTS_DIR / "dynamic_k_analysis.png", dpi=150)
    plt.show()
    print(f"Saved to {RESULTS_DIR / 'dynamic_k_analysis.png'}")

    # Summary stats
    print("\n=== Dynamic K summary ===")
    for drafter in ["cnn", "hmm"]:
        sub = dyn_df[dyn_df["drafter"] == drafter].copy()
        if len(sub) == 0:
            continue
        sub["mean_k"] = sub["k_history"].apply(
            lambda h: np.mean(h) if isinstance(h, list) and h else float("nan")
        )
        print(f"\n{drafter.upper()}:")
        print(f"  Mean K (across proteins): {sub['mean_k'].mean():.2f}  "
              f"(K_init=5, range [{sub['mean_k'].min():.1f}, {sub['mean_k'].max():.1f}])")
        print(f"  Speedup: mean={sub['speedup'].mean():.2f}x  median={sub['speedup'].median():.2f}x")
        print(f"  Acceptance rate: {sub['acceptance_rate'].mean():.3f}")

In [ ]:
#@title Save all results to Drive. { display-mode: "form" }

# Save final state
save_spec_checkpoint({"results": results, "completed": list(completed)})

# Copy key outputs to Drive for persistence
if USE_DRIVE:
    import shutil
    drive_results = Path(DRIVE_ROOT) / "spec_decode_results"
    drive_results.mkdir(exist_ok=True)
    for f in RESULTS_DIR.glob("*"):
        if f.is_file():
            shutil.copy(f, drive_results / f.name)
    print(f"Results copied to Drive: {drive_results}")
else:
    print(f"Results at: {RESULTS_DIR}")

print("\nDone! Key files:")
print(f"  - spec_decode_results.csv (all timings)")
print(f"  - predicted_vs_measured.csv (α comparison)")
print(f"  - spec_decode_plots.png")